# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Scope note

This playbook is built on `work/baseline_action_score.csv` — the Week 4 baseline rule's
output, the same file used in `w03_feature_leakage_check` and `w04_signal_audit`. There is
no Week 5/6 model output to layer on top of it yet (see the note at the end of this
notebook): the ranked queue below is the honest baseline queue, not a model-improved one.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd

df = pd.read_csv("work/baseline_action_score.csv")

queue = (
    df[df["action"] == "REVIEW_REFRESH"]
    .sort_values("baseline_score", ascending=False)
    .loc[:, ["rank", "content_hash_id", "baseline_score", "reason_code",
             "march_impressions", "staleness_days"]]
)

print("Pages in the REVIEW_REFRESH queue:", len(queue))
print("Score range:", queue['baseline_score'].min(), "to", queue['baseline_score'].max())
queue.head(20)


Pages in the REVIEW_REFRESH queue: 10189
Score range: 70.0 to 95.70413143835457


,rank,content_hash_id,baseline_score,reason_code,march_impressions,staleness_days
0,1,content_42ce26be1ec6be00,95.704131,STALE_HIGH_OPPORTUNITY,4411,264.0
1,2,content_bea86ce3455100b0,94.513173,STALE_HIGH_OPPORTUNITY,3670,232.0
2,3,content_097459d155cccb26,94.225199,STALE_HIGH_OPPORTUNITY,37930,124.0
3,4,content_f2df5a8a9057783e,94.209147,STALE_HIGH_OPPORTUNITY,35980,124.0
4,5,content_ac4e2d9d3bbb06de,94.184859,STALE_HIGH_OPPORTUNITY,33348,124.0
5,6,content_66d1fffc91f4f029,94.162683,STALE_HIGH_OPPORTUNITY,31038,124.0
6,7,content_9598a57544925111,94.018801,STALE_HIGH_OPPORTUNITY,24066,123.0
7,8,content_b956947c822af734,94.001325,STALE_HIGH_OPPORTUNITY,22502,124.0
8,9,content_19daa2f24df1882d,93.986588,STALE_HIGH_OPPORTUNITY,4968,176.0
9,10,content_0d2aaf57d7146812,93.935587,STALE_HIGH_OPPORTUNITY,21060,123.0


**Top of the queue, in words a human trusts:** every page here carries the reason code
`STALE_HIGH_OPPORTUNITY` — it combines observed March 2026 search impressions with a
recorded staleness (days since last update). The queue is sorted by `baseline_score`
descending, so row 1 is the single highest-priority page for human review this cycle.
As the signal audit above shows, impressions do almost all of the ranking work here —
treat the ranking as an impressions-led opportunity list with a staleness tie-breaker,
not a balanced staleness score.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
print("Who uses this: a content or search-ops reviewer with limited weekly review capacity.")
print("What for: deciding which pages to open FIRST for a manual refresh/expand/protect/")
print("prune/monitor decision — the queue is a triage order, not a verdict.")
print()
print("Where it stops being valid:")
print(f"- {(df['staleness_days'].isna().mean()):.1%} of all pages have no recorded update date,")
print("  so they cannot currently reach this queue at all, however much traffic they get")
print("  (see the flag-linked test in w04_signal_audit).")
print("- The score reflects March 2026 only; it says nothing about why a page's traffic")
print("  moved, and nothing about future performance if refreshed.")


Who uses this: a content or search-ops reviewer with limited weekly review capacity.
What for: deciding which pages to open FIRST for a manual refresh/expand/protect/
prune/monitor decision — the queue is a triage order, not a verdict.

Where it stops being valid:
- 88.5% of all pages have no recorded update date,
  so they cannot currently reach this queue at all, however much traffic they get
  (see the flag-linked test in w04_signal_audit).
- The score reflects March 2026 only; it says nothing about why a page's traffic
  moved, and nothing about future performance if refreshed.


**Intended use:** triage input for a human reviewer with limited weekly capacity, to
decide which pages to open first — never an automatic refresh/prune/protect decision.
**Limits:** valid only for the March 2026 window; silent on causation; structurally blind
to the ~88% of pages with no recorded update date.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
print("Human review checklist before acting on any REVIEW_REFRESH row:")
print("1. Confirm march_impressions reflects a real, non-bot-inflated audience (spot-check GSC).")
print("2. Confirm the page's actual content/business context — a rising doc rewrite,")
print("   a seasonal page, or a page slated for deprecation reads very differently even")
print("   at the same score.")
print("3. Confirm staleness_days reflects a genuine content edit date, not a template/")
print("   metadata timestamp bump.")
print()
print("No-go list — never automate directly from this score:")
print("- Never auto-publish, auto-rewrite, or auto-prune a page from baseline_score alone.")
print("- Never treat a MONITOR label as 'confirmed fine' for pages with missing staleness —")
print("  it may mean 'unscored', not 'healthy' (see signal audit).")
print("- Never present baseline_score as a causal prediction of future traffic change.")


Human review checklist before acting on any REVIEW_REFRESH row:
1. Confirm march_impressions reflects a real, non-bot-inflated audience (spot-check GSC).
2. Confirm the page's actual content/business context — a rising doc rewrite,
   a seasonal page, or a page slated for deprecation reads very differently even
   at the same score.
3. Confirm staleness_days reflects a genuine content edit date, not a template/
   metadata timestamp bump.

No-go list — never automate directly from this score:
- Never auto-publish, auto-rewrite, or auto-prune a page from baseline_score alone.
- Never treat a MONITOR label as 'confirmed fine' for pages with missing staleness —
  it may mean 'unscored', not 'healthy' (see signal audit).
- Never present baseline_score as a causal prediction of future traffic change.


**Human review, always:** audience quality, business/editorial context, and whether
`staleness_days` reflects a real edit — none of these are checkable from the score alone.
**Never automate:** publishing, rewriting, or pruning a page purely because of its
`baseline_score`, or treating `MONITOR` as a clean bill of health for pages the rule
never actually got to evaluate.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
print("Monitoring / retrain triggers — what would tell us this queue has gone stale:")
print(f"- staleness_days observed-rate is currently {df['staleness_days'].notna().mean():.1%};")
print("  if that rate changes materially, the REVIEW_REFRESH queue's coverage has shifted")
print("  and thresholds should be re-checked.")
print(f"- The REVIEW_REFRESH cut point is currently baseline_score >= 70.0")
print("  ({} of {} pages, {:.1%}); if the share crossing that cut drifts far from that,".format(
    (df['baseline_score'] >= 70).sum(), len(df), (df['baseline_score'] >= 70).mean()))
print("  re-derive the threshold from the new month's distribution rather than reusing 70.0.")
print("- Any month where march_impressions distribution shape changes sharply (a big jump")
print("  or crash in the zero-impressions share, currently {:.1%}) is a signal to re-run".format(
    (df['march_impressions']==0).mean()))
print("  this whole audit before trusting the queue.")


Monitoring / retrain triggers — what would tell us this queue has gone stale:
- staleness_days observed-rate is currently 11.5%;
  if that rate changes materially, the REVIEW_REFRESH queue's coverage has shifted
  and thresholds should be re-checked.
- The REVIEW_REFRESH cut point is currently baseline_score >= 70.0
  (10189 of 331437 pages, 3.1%); if the share crossing that cut drifts far from that,
  re-derive the threshold from the new month's distribution rather than reusing 70.0.
- Any month where march_impressions distribution shape changes sharply (a big jump
  or crash in the zero-impressions share, currently 46.7%) is a signal to re-run
  this whole audit before trusting the queue.


**Retrain/re-audit triggers:** a material shift in the staleness-observed rate (currently
~11.5%), a drift in the share of pages crossing the score-70 cut (currently ~3.1%), or a
shift in the zero-impressions share (currently ~46.7%) should all prompt re-deriving
thresholds from the new month's data rather than reusing March 2026's numbers.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import os

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/review_queue_week7.csv"
queue.to_csv(out_path, index=False)

print("Exported:", out_path)
print("Rows:", len(queue))
print("Exists on disk:", os.path.exists(out_path))
print("File size (bytes):", os.path.getsize(out_path))


Exported: work/outputs/review_queue_week7.csv
Rows: 10189
Exists on disk: True
File size (bytes): 820453


**Honest gap, flagged rather than papered over:** this notebook only reproduces and
operationalizes the Week 4 baseline queue — there is no Week 5 model output to compare it
against yet, because `w05_model` and `w06_validation_audit` need the fuller feature set and
target label from `w03_data_contract` (`avg_search_position`, `march_sessions`,
`engagement_rate`, `march_clicks`, `is_declining_proxy`), which live in the FlyRank Hugging
Face warehouse and aren't in any file uploaded to this conversation so far. Once that
feature/target file is available, this playbook should be re-run against the Week 5/6
model's ranked output instead of the raw baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.